# Exercise 5: How reliable is the result? Validation over time
**Internet of Production – AI, week 2** · about 35 min · work in pairs

**Thread:** Exercise 4 gave one number per model. But the data are a **time series with only 34 blockages**. Can you trust a single split? In this exercise you check how much the result depends on how you validate.

**Questions**
1. Why did we split by time and not randomly? What changes if we split randomly?
2. How much does the result vary if the split moves through time?
3. Do additional features (temperatures) help, or do they only fit noise?

Every `___` is a gap.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes: warn 6 hours ahead
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 of week 1 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values)
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages, idx)
    mtb = np.where(nxt < len(blockages), (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    p['Temp_Extruder_m'] = p['Temp_Extruder'].rolling(120).mean()
    p['Temp_Bed_m'] = p['Temp_Bed'].rolling(120).mean()
    return p.dropna(subset=FEATURES + ['Temp_Extruder_m']).iloc[::10], pd.to_datetime(blockages)

def report(name, y_true, y_pred, score=None):
    auc = f'   AUC {roc_auc_score(y_true, score):5.3f}' if score is not None else ''
    print(f'{name:36s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}{auc}')

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)                      # split by time: first 70 % train, last 30 % test
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[FEATURES], train['y'], test[FEATURES], test['y']
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1; train {len(train)}, test {len(test)}")

from sklearn.model_selection import train_test_split, TimeSeriesSplit

## Step 1: Random split versus time split
Neighbouring samples are only 10 minutes apart and almost identical, and their labels are the same most of the time. With a **random** split, a test sample then has a near-copy in the training set. Compare three models with both splits (recall on the test set).

In [ ]:
models = {
    '1-nearest neighbour': lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
    'tree, no limit': lambda: DecisionTreeClassifier(random_state=0),
    'tree, depth 3, balanced': lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, class_weight='balanced', random_state=0),
    'KNN, k = 25': lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
}
Xr_train, Xr_test, yr_train, yr_test = train_test_split(data[FEATURES], data['y'], test_size=0.3, random_state=0, stratify=data['y'])

rows = []
for name, make in models.items():
    m_time = make().fit(X_train, y_train)
    m_rand = make().fit(Xr_train, yr_train)
    rows.append({'model': name,
                 'recall, time split': recall_score(y_test, m_time.predict(X_test)),
                 'recall, random split': recall_score(yr_test, m_rand.predict(Xr_test)),
                 'precision, time split': precision_score(y_test, m_time.predict(X_test), zero_division=0),
                 'precision, random split': precision_score(yr_test, m_rand.predict(Xr_test), zero_division=0)})
display(pd.DataFrame(rows).set_index('model').round(3))

**For which models does the random split look better than the time split? Why do flexible models profit most? What would a customer expect from the result?**

The random split looks better for the flexible models (1-NN, unrestricted tree: recall about 0.58 to 0.60 against 0.45 to 0.53 with the time split) because they find a near-copy of each test sample in the training data. Smooth models (KNN with k = 25, the small tree) change little (+0.00 and +0.04). In operation the printer only delivers **future** data, so the time split is the honest test; a random split would promise a quality that the customer does not get.

## Step 2: Move the split through time
`TimeSeriesSplit` trains on the past and tests on the next block, five times with a growing training set. Count how many **blockages** each test block contains.

In [ ]:
def evaluate_over_time(make, features, n_splits=5):
    rows = []
    for i, (a, bb) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(data), 1):
        tr, te = data.iloc[a], data.iloc[bb]
        m = make().fit(tr[features], tr['y'])
        n_blk = int(((blockages >= te.index[0]) & (blockages <= te.index[-1])).sum())
        rows.append({'fold': i, 'test from': te.index[0].date(), 'blockages in test': n_blk,
                     'recall': recall_score(te['y'], m.predict(te[features])),
                     'precision': precision_score(te['y'], m.predict(te[features]), zero_division=0),
                     'AUC': roc_auc_score(te['y'], m.predict_proba(te[features])[:, 1])})
    return pd.DataFrame(rows).set_index('fold')

folds = evaluate_over_time(models['tree, depth 3, balanced'], FEATURES)
display(folds.round(3))
print('mean recall', folds['recall'].mean().round(3), ' min', folds['recall'].min().round(3), ' max', folds['recall'].max().round(3))

**How much does the recall vary between the blocks, and how many blockages are behind each block? What does that mean for the sentence "the model finds 80 % of the blockages"?**

The recall varies between about 0.69 and 0.99, with only 3 to 10 blockages per block. One more or less missed blockage changes the value by 10 to 30 percentage points. A statement such as "80 %" therefore has an uncertainty of at least +/- 15 percentage points and should always be given with the number of events behind it.

## Step 3: Do temperatures help?
Add the mean extruder and bed temperature of the last 2 hours as features and repeat the evaluation over time. Judge the change with the variation you just saw.

In [ ]:
features_t = FEATURES + ['Temp_Extruder_m', 'Temp_Bed_m']
folds_t = evaluate_over_time(models['tree, depth 3, balanced'], features_t)
cmp = pd.DataFrame({'AUC energy only': folds['AUC'], 'AUC + temperatures': folds_t['AUC']})
cmp['difference'] = cmp['AUC + temperatures'] - cmp['AUC energy only']
display(cmp.round(3)); print(cmp.mean().round(3).to_dict())

# physical plausibility: mean temperatures in the hours before a blockage
dd = data.copy()
dd['group'] = np.where(dd['y'] == 1, 'within 6 h before blockage', 'other')
display(dd.groupby('group')[['E120', 'Temp_Extruder_m', 'Temp_Bed_m']].mean().round(3))

**Would you add the temperatures to the model? Consider the size of the gain, the number of events, and whether there is a physical explanation.**

The AUC rises by 0.02 to 0.06 in four of five blocks (mean 0.89 to 0.92), but the gain is as small as the differences between the blocks, and 34 events are few. The energy has a physical explanation (more friction in a clogged nozzle), and its rise is steady towards the blockage. The bed temperature differs only by about 0.08 °C and not steadily with the time to blockage, so there is no explanation yet. Recommendation: do not add it to the product; keep it as a hypothesis and test it on printer 2 (transfer test, week 3).

## Step 4: Summary

**In four sentences: what would you tell the head of pre-production about the quality of the warning, and what do you recommend for the next steps (data, validation, model)?**

The warning finds roughly 55 to 80 % of the extruder blockages hours ahead, depending on the threshold, at the price of false alarms; this is a statement from few events and has an uncertainty of about 15 percentage points. A simple, readable rule on the 2-hour mean of the energy is as good as KNN and can be explained to the shop floor. Next, the rule must be tested on another printer (week 3) and on new data in operation, with the validation always going forward in time. More features only after a physical reason is known.